# Download KodeKloud Course into Google Drive

## Disclaimer
Please read the following disclaimer carefully before using the Downloader CLI Tool.

- This is intended for personal use only. By using this notebook, you agree to use it at your own risk and assume full responsibility for any consequences that may arise from its use. The developers and contributors of this notebook are not responsible for any damages or losses that may occur from its use.

- The use of this tool to download courses is for educational purposes only. You must have the proper authorization or active subscription from the course provider to access the content legally.

- It is strictly prohibited to distribute or share the downloaded content through any means, including but not limited to uploading to file-sharing platforms, torrent sites, or any other form of digital or physical distribution. Doing so is a violation of copyright laws and may result in legal consequences.

## Step 1: Install kodekloud-downloader

Install the updated downloader directly from the fork repository, along with browser support for token exchange.

> **Note**: FFmpeg is already pre-installed on Google Colab runtimes.

In [ ]:
# Install downloader with browser dependencies from fork
!pip install -U "kodekloud-downloader[browser] @ git+https://github.com/NovoG93/kodekloud-downloader.git"

# Install Chromium for Playwright (required if using automated cookie-to-token exchange)
!playwright install --with-deps chromium

## Step 2: Mount Google Drive

Mount your Google Drive to save courses directly into your cloud storage.

> **Make sure this is the same Google account where this notebook is running.**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

### Upload Cookie File

- Sign in to [kodekloud.com](https://learn.kodekloud.com)
- Export cookies using an extension such as [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc/related)
- Make sure the exported cookie file contains `_secure-user-session`.
- The downloader will launch headless Chromium to automatically exchange your session cookie for a Firebase ID token.

In [ ]:
from google.colab import files

uploaded = files.upload()
cookie_file_name = list(uploaded.keys())[0]
print(f"Uploaded cookie file: {cookie_file_name}")

In [ ]:
# Download courses to Google Drive using cookie file
!kodekloud dl -c "$cookie_file_name" -o "/content/drive/MyDrive"

In [ ]:
import os

# Check the contents of the MyDrive folder to verify the correct directory name
drive_path = '/content/drive/MyDrive'
if os.path.exists(drive_path):
    print(f'Contents of {drive_path}:')
    print(os.listdir(drive_path))
else:
    print(f'Drive path {drive_path} does not exist. Check if drive is mounted.')

In [ ]:
from google.colab import drive
import os

# Attempting to mount Google Drive again to access the downloaded files
try:
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        print('Drive successfully mounted and MyDrive is accessible.')
    else:
        print('Drive mount command finished but MyDrive not found.')
except Exception as e:
    print(f'Failed to mount drive: {e}')

In [ ]:
import os
from pathlib import Path
import time

base_path = Path('/content/drive/MyDrive/KodeKloud')
current_time = time.time()
session_threshold_seconds = 86400  # 24 hours to be safe given metadata sync

newly_downloaded_videos = []

if base_path.exists():
    print(f'Scanning {base_path} for video files...')
    for file_path in base_path.rglob('*'):
        if file_path.suffix.lower() in ['.mkv', '.mp4']:
            file_mtime = file_path.stat().st_mtime
            # We check if it was modified recently
            if (current_time - file_mtime) <= session_threshold_seconds:
                newly_downloaded_videos.append(str(file_path))

print(f'Found {len(newly_downloaded_videos)} video files:')
for video in newly_downloaded_videos:
    print(video)

In [ ]:
import subprocess
import re
from pathlib import Path

converted_mp3_files = []

def pad_leading_number(name):
    """Detects leading numbers and zero-pads them so '1' becomes '01'."""
    match = re.match(r'^(\d+)(\s*-\s*.*)', name)
    if match:
        return f"{match.group(1).zfill(2)}{match.group(2)}"
    return name

print(f"Starting conversion of {len(newly_downloaded_videos)} files...")

for video_path_str in newly_downloaded_videos:
    video_path = Path(video_path_str)

    # 1. Get the original chapter folder name (e.g., "1 - Introduction")
    chapter_folder_name = video_path.parent.name

    # 2. Get the main course folder (e.g., "Prep Course GitOps Certified Associate CGOA")
    course_base_dir = video_path.parent.parent

    # 3. Build the new target directory path: [Course] / "mp3s" / [Chapter]
    target_dir = course_base_dir / "mp3s" / chapter_folder_name

    # 4. Create this new directory structure if it doesn't exist yet
    target_dir.mkdir(parents=True, exist_ok=True)

    # 5. Zero-pad the file name so the tracks play in order within the folder
    file_stem = pad_leading_number(video_path.stem)
    new_filename = f"{file_stem}.mp3"

    # 6. Final output path
    mp3_path = target_dir / new_filename

    print(f"Converting: {video_path.name} -> {mp3_path}")

    # Construct ffmpeg command
    command = [
        'ffmpeg',
        '-i', str(video_path),
        '-vn',
        '-acodec', 'libmp3lame',
        '-y',
        str(mp3_path)
    ]

    try:
        # Execute the command
        result = subprocess.run(command, capture_output=True, text=True)

        if result.returncode == 0:
            converted_mp3_files.append(str(mp3_path))
        else:
            print(f"Error converting {video_path.name}: {result.stderr}")
    except Exception as e:
        print(f"Failed to process {video_path.name}: {e}")

print(f"\nBatch conversion complete. Total MP3s created: {len(converted_mp3_files)}")